In [ ]:
import os
import shutil
import urllib.request
import pandas as pd
import numpy as np
from lightgbm import LGBMClassifier
import gc
import warnings
warnings.filterwarnings("ignore")

# =============================================================================
# 0. FUNCIONES DE OPTIMIZACIÓN DE MEMORIA
# =============================================================================
def reducir_memoria(df):
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != object:
            if str(col_type)[:3] == 'int':
                df[col] = pd.to_numeric(df[col], downcast='integer')
            else:
                df[col] = pd.to_numeric(df[col], downcast='float')
    return df

# =============================================================================
# 1. MONTAJE DE DRIVE Y DESCARGA CONDICIONAL
# =============================================================================
try:
    from google.colab import drive
    if not os.path.exists('/content/.drive'):
        drive.mount('/content/.drive')
except ImportError:
    pass

os.makedirs("/content/.drive/My Drive/dmeyf", exist_ok=True)
os.makedirs("/content/buckets/b1/datasets", exist_ok=True)
os.makedirs("/content/datasets", exist_ok=True)

if not os.path.exists("/content/buckets/b1"):
    try:
        os.symlink("/content/.drive/My Drive/dmeyf", "/content/buckets/b1")
    except:
        pass

file_drive = "/content/buckets/b1/datasets/competencia_01_crudo.csv"
file_local = "/content/datasets/competencia_01_crudo.csv"
url = "https://storage.googleapis.com/open-courses/dmeyf2026-9c6f/competencia_01_crudo.csv"

if not os.path.isfile(file_drive):
    urllib.request.urlretrieve(url, file_drive)

if not os.path.isfile(file_local):
    shutil.copy(file_drive, file_local)

# =============================================================================
# 2. FUNCIÓN: RANKING CON CERO FIJO IN-PLACE
# =============================================================================
def aplicar_rank_cero_fijo_inplace(df, columnas_a_rankear):
    def escalar_rangos(s, val_min, val_max):
        if s.empty:
            return s
        r = s.rank(method='dense')
        r_min, r_max = r.min(), r.max()
        if r_min == r_max:
            return pd.Series(val_max, index=s.index)
        p = (r - r_min) / (r_max - r_min)
        return val_min + p * (val_max - val_min)

    for col in columnas_a_rankear:
        mask_pos = df[col] > 0
        mask_neg = df[col] < 0

        if mask_pos.any():
            df.loc[mask_pos, col] = df.loc[mask_pos].groupby('foto_mes')[col].transform(lambda x: escalar_rangos(x, 0.001, 1.0))
        if mask_neg.any():
            df.loc[mask_neg, col] = df.loc[mask_neg].groupby('foto_mes')[col].transform(lambda x: escalar_rangos(x, -1.0, -0.001))
    return df

# =============================================================================
# 3. CARGA, COMPRESIÓN Y CREACIÓN DE LA CLASE BINARIA
# =============================================================================
print("Cargando dataset...")
dataset = pd.read_csv(file_local)
dataset = reducir_memoria(dataset)

dataset.loc[dataset['foto_mes'] == 202105, 'ccajas_depositos'] = np.nan

dataset['periodo0'] = (dataset['foto_mes'] // 100) * 12 + (dataset['foto_mes'] % 100)
dataset.sort_values(['numero_de_cliente', 'periodo0'], inplace=True)

dataset['periodo1'] = dataset.groupby('numero_de_cliente')['periodo0'].shift(-1)
dataset['periodo2'] = dataset.groupby('numero_de_cliente')['periodo0'].shift(-2)

periodo_ultimo = dataset['periodo0'].max()
periodo_anteultimo = periodo_ultimo - 1

dataset['clase_binaria'] = np.nan
dataset.loc[dataset['periodo0'] < periodo_anteultimo, 'clase_binaria'] = 'CONTINUA'

mask_baja1 = (dataset['periodo0'] < periodo_ultimo) & (dataset['periodo1'].isna() | (dataset['periodo0'] + 1 < dataset['periodo1']))
mask_baja2 = (dataset['periodo0'] < periodo_anteultimo) & (dataset['periodo0'] + 1 == dataset['periodo1']) & (dataset['periodo2'].isna() | (dataset['periodo0'] + 2 < dataset['periodo2']))

dataset.loc[mask_baja1 | mask_baja2, 'clase_binaria'] = 'BAJA'
dataset['target'] = np.where(dataset['clase_binaria'] == 'BAJA', 1, np.where(dataset['clase_binaria'] == 'CONTINUA', 0, np.nan))
dataset.drop(columns=['periodo0', 'periodo1', 'periodo2'], inplace=True)
gc.collect()

# =============================================================================
# 4. APLICACIÓN DE RANKING
# =============================================================================
print("Aplicando Ranking Cero Fijo...")
features_base = [c for c in dataset.columns if c not in ['numero_de_cliente', 'foto_mes', 'clase_binaria', 'target']]
dataset = aplicar_rank_cero_fijo_inplace(dataset, features_base)
gc.collect()

# =============================================================================
# 5. FEATURE ENGINEERING: LAGS Y DELTAS
# =============================================================================
print("Calculando Lags y Deltas...")
dataset.sort_values(['numero_de_cliente', 'foto_mes'], inplace=True)
dataset['periodo_temp'] = (dataset['foto_mes'] // 100) * 12 + (dataset['foto_mes'] % 100)

for col in features_base:
    dataset[f'{col}_LAG1'] = dataset.groupby('numero_de_cliente')[col].shift(1).astype('float32')
    dataset[f'{col}_LAG2'] = dataset.groupby('numero_de_cliente')[col].shift(2).astype('float32')

    periodo_lag1 = dataset.groupby('numero_de_cliente')['periodo_temp'].shift(1)
    periodo_lag2 = dataset.groupby('numero_de_cliente')['periodo_temp'].shift(2)

    dataset.loc[dataset['periodo_temp'] - periodo_lag1 != 1, f'{col}_LAG1'] = np.nan
    dataset.loc[dataset['periodo_temp'] - periodo_lag2 != 2, f'{col}_LAG2'] = np.nan

    dataset[f'{col}_DELTA-LAG1'] = (dataset[col] - dataset[f'{col}_LAG1']).astype('float32')
    dataset[f'{col}_DELTA-LAG2'] = (dataset[col] - dataset[f'{col}_LAG2']).astype('float32')

dataset.drop(columns=['periodo_temp'], inplace=True)
features_totales = [c for c in dataset.columns if c not in ['numero_de_cliente', 'foto_mes', 'clase_binaria', 'target']]
gc.collect()

import os
import shutil
import urllib.request
import pandas as pd
import numpy as np
from lightgbm import LGBMClassifier
import gc
import warnings
warnings.filterwarnings("ignore")

# =============================================================================
# 6. SPLIT TEMPORAL PARA VALIDACIÓN OFFLINE (Train: 2103 a 2105 | Val: 2106)
# =============================================================================
print("Preparando particiones para validación offline...")
meses_train_val = [202103, 202104, 202105]
train_mask_val = dataset['foto_mes'].isin(meses_train_val) & dataset['target'].notna()
X_train_val = dataset.loc[train_mask_val, features_totales]
y_train_val = dataset.loc[train_mask_val, 'target']

val_mask = dataset['foto_mes'] == 202106
X_val = dataset.loc[val_mask, features_totales]
y_val = dataset.loc[val_mask, 'target']
clientes_val = dataset.loc[val_mask, 'numero_de_cliente'].values

del dataset
gc.collect()

# =============================================================================
# 7. ENTRENAMIENTO RÁPIDO (3 SEMILLAS)
# =============================================================================
n_semillas_val = 3
np.random.seed(331613)
semillas_val = np.random.randint(10000, 999999, size=n_semillas_val)

params = {
    'n_estimators': 1250, 'num_leaves': 700, 'min_sum_hessian_in_leaf': 0.003,
    'learning_rate': 0.005, 'min_child_samples': 0, 'colsample_bytree': 0.50,
    'max_depth': -1, 'min_split_gain': 0.0, 'reg_alpha': 0.0, 'reg_lambda': 0.0,
    'max_bin': 31, 'subsample': 1.0, 'is_unbalance': False, 'scale_pos_weight': 1.0,
    'extra_trees': False, 'boosting_type': 'gbdt', 'objective': 'binary',
    'feature_pre_filter': False, 'deterministic': True, 'verbosity': -100, 'n_jobs': -1
}

predicciones_val = np.zeros(len(X_val))
print(f"Iniciando entrenamiento del ensamble de validación ({n_semillas_val} modelos)...")

for i, semilla in enumerate(semillas_val, 1):
    print(f"Entrenando modelo {i}/{n_semillas_val}...")
    params['random_state'] = int(semilla)
    clf = LGBMClassifier(**params)
    clf.fit(X_train_val, y_train_val)
    predicciones_val += clf.predict_proba(X_val)[:, 1]
    del clf
    gc.collect()

probabilidades_val = predicciones_val / n_semillas_val

# =============================================================================
# 8. CÁLCULO DE GANANCIA Y PUNTO DE CORTE ÓPTIMO
# =============================================================================
# Matriz de negocio (Ajustá estos valores si tu cursada maneja otros montos)
ganancia_acierto = 273000
costo_estimulo = 7000

df_eval = pd.DataFrame({
    'numero_de_cliente': clientes_val,
    'probabilidad': probabilidades_val,
    'target': y_val.values
})

# Ordenar por probabilidad descendente
df_eval = df_eval.sort_values(by='probabilidad', ascending=False).reset_index(drop=True)

# Calcular ganancia fila por fila: Si es BAJA (1) ganamos 273k, si es CONTINUA (0) perdemos 7k
df_eval['ganancia_individual'] = np.where(df_eval['target'] == 1, ganancia_acierto, -costo_estimulo)

# Acumular la ganancia a medida que bajamos en la lista
df_eval['ganancia_acumulada'] = df_eval['ganancia_individual'].cumsum()

# Encontrar el índice donde la ganancia es máxima (sumamos 1 porque el índice arranca en 0)
corte_optimo = df_eval['ganancia_acumulada'].idxmax() + 1
ganancia_maxima = df_eval['ganancia_acumulada'].max()

print("\n" + "="*50)
print(f"RESULTADOS DE LA VALIDACIÓN OFFLINE (Mes 202106):")
print(f"Ganancia Máxima Alcanzada: ${ganancia_maxima:,.2f}")
print(f"Corte Óptimo Sugerido: {corte_optimo} envíos")
print("="*50)

Mounted at /content/.drive
Cargando dataset...
Aplicando Ranking Cero Fijo...
Calculando Lags y Deltas...
Preparando particiones para validación offline...
Iniciando entrenamiento del ensamble de validación (3 modelos)...
Entrenando modelo 1/3...
Entrenando modelo 2/3...
